In [ ]:
import sys
import os

# Ensure current directory is on python path
sys.path.insert(0, os.path.abspath("."))

from src.models.entities import Patient, BiometricRecord, InvalidBiometricError
from src.engines.risk_models import (
    HypertensionRiskModel,
    DiabetesRiskModel,
    CardiovascularRiskModel,
    CompositeRiskEvaluator
)
from src.data.pipeline import CohortDataLoader
from src.analytics.visualizer import ClinicalVisualizer

print("HealthGuard Analytics loaded successfully.")

In [ ]:
I use CohortDataLoader to read the cohort CSV file. It automatically creates Patient and BiometricRecord objects. If any record has physically impossible numbers (like diastolic BP higher than systolic BP), it catches InvalidBiometricError and logs it instead of crashing

In [ ]:
# Load cohort data
patients, error_logs = CohortDataLoader.load_from_csv("data/sample_patients.csv")

print(f"Successfully loaded {len(patients)} valid patient records.")
print("\nValidation Error Logs (Handling corrupted data gracefully):")
for err in error_logs:
    print(" -", err)

In [ ]:
I initialize our concrete risk models (HypertensionRiskModel, DiabetesRiskModel, CardiovascularRiskModel) and pass them into the CompositeRiskEvaluator. The evaluator processes each model uniformly through polymorphism without needing if/else checks.

In [ ]:
# Initialize models and evaluator
evaluator = CompositeRiskEvaluator([
    HypertensionRiskModel(),
    DiabetesRiskModel(),
    CardiovascularRiskModel()
])

# Build summary DataFrame
df_cohort = ClinicalVisualizer.build_summary_dataframe(patients, evaluator)
display(df_cohort[["patient_id", "name", "age", "gender", "systolic_bp", "glucose", "hypertension_risk", "diabetes_risk", "cvd_risk"]])

In [ ]:
# Render the clinical visualization dashboard
ClinicalVisualizer.plot_cohort_summary(df_cohort)

In [ ]:
#Clinical Summary & Insights
## 1. Hypertension Risk: Older patients frequently exhibit Stage 1 and Stage 2 hypertension, with several flagged for immediate hypertensive care.
## 2. Comorbidities: Several patients with elevated systolic BP also present with elevated fasting glucose (> 7.0mmol/L), putting them in the High CVD Risk category.
## 3.Conclusion: The Object-Oriented design provides modular, safe, and easily extensible health screening support